# 04 · Técnicas sobre arreglos: prefijos, dos punteros y ventana deslizante

Tópico 04 del temario. Tres técnicas que convierten soluciones `O(n²)` en `O(n)`.

## 1. Sumas de prefijos

**Idea:** guardar `pref[i]` = suma de los primeros `i` elementos. Así la suma de cualquier rango sale **restando dos valores** en `O(1)`.

```
arr  =   3  1  4  1  5
pref = 0  3  4  8  9 14        (pref[0] = 0)

suma de arr[l..r]  (índices desde 0, inclusive) = pref[r + 1] - pref[l]
```

In [ ]:
from itertools import accumulate

arr = [3, 1, 4, 1, 5]

pref = [0]
for x in arr:
    pref.append(pref[-1] + x)
assert pref == [0, 3, 4, 8, 9, 14]

# Con la librería estándar
assert [0] + list(accumulate(arr)) == pref


def suma_rango(pref, l, r):
    return pref[r + 1] - pref[l]


assert suma_rango(pref, 1, 3) == 1 + 4 + 1
assert suma_rango(pref, 0, 4) == 14

### ¿Cuándo sirve?

Cuando te hacen **muchas consultas** de suma sobre rangos. Sin prefijos cada consulta cuesta `O(n)`; con prefijos, `O(1)` después de un precálculo `O(n)`.

In [ ]:
def consultas_suma(arr, consultas):
    pref = [0]
    for x in arr:
        pref.append(pref[-1] + x)
    return [pref[r + 1] - pref[l] for l, r in consultas]


assert consultas_suma([3, 1, 4, 1, 5], [(0, 4), (1, 3), (2, 2)]) == [14, 6, 4]

### Variante: contar cuántos subarreglos suman `k`

Con prefijos y un diccionario: si `pref[j] - pref[i] = k` entonces `pref[i] = pref[j] - k`.

In [ ]:
def subarreglos_con_suma(arr, k):
    visto = {0: 1}          # cuántas veces apareció cada prefijo
    suma = 0
    total = 0
    for x in arr:
        suma += x
        total += visto.get(suma - k, 0)
        visto[suma] = visto.get(suma, 0) + 1
    return total


assert subarreglos_con_suma([1, 1, 1], 2) == 2
assert subarreglos_con_suma([3, 4, 7, 2, -3, 1, 4, 2], 7) == 4

### Prefijos en 2D

`pref[i][j]` = suma del rectángulo desde `(0,0)` hasta `(i-1, j-1)`.

In [ ]:
def prefijos_2d(m):
    f, c = len(m), len(m[0])
    p = [[0] * (c + 1) for _ in range(f + 1)]
    for i in range(f):
        for j in range(c):
            p[i + 1][j + 1] = m[i][j] + p[i][j + 1] + p[i + 1][j] - p[i][j]
    return p


def suma_rectangulo(p, f1, c1, f2, c2):
    """Suma de m[f1..f2][c1..c2], ambos extremos incluidos."""
    return p[f2 + 1][c2 + 1] - p[f1][c2 + 1] - p[f2 + 1][c1] + p[f1][c1]


m = [[1, 2, 3], [4, 5, 6], [7, 8, 9]]
p = prefijos_2d(m)
assert suma_rectangulo(p, 0, 0, 2, 2) == 45
assert suma_rectangulo(p, 1, 1, 2, 2) == 5 + 6 + 8 + 9

### Diferencias (la idea inversa)

Para **sumar `v` a todo un rango** muchas veces: suma `v` en `l` y resta `v` en `r + 1`; al final haz la suma acumulada.

In [ ]:
def aplicar_rangos(n, operaciones):
    d = [0] * (n + 1)
    for l, r, v in operaciones:
        d[l] += v
        d[r + 1] -= v
    return list(accumulate(d))[:n]


assert aplicar_rangos(5, [(0, 2, 3), (1, 4, 2)]) == [3, 5, 5, 2, 2]

## 2. Dos punteros

**Idea:** dos índices que se mueven por el arreglo, normalmente de forma coordinada, en lugar de probar todos los pares.

### Variante A: uno en cada extremo (arreglo **ordenado**)

Problema: ¿hay dos números que sumen `k`?

- Si `arr[i] + arr[j] < k`, necesitas más: avanza `i`.
- Si es mayor, necesitas menos: retrocede `j`.

In [ ]:
def dos_suma_ordenado(arr, k):
    i, j = 0, len(arr) - 1
    while i < j:
        s = arr[i] + arr[j]
        if s == k:
            return True
        if s < k:
            i += 1
        else:
            j -= 1
    return False


assert dos_suma_ordenado([1, 3, 4, 6, 8], 10)          # 4 + 6
assert not dos_suma_ordenado([1, 3, 4, 6, 8], 20)

### Variante B: ambos avanzan en el mismo sentido

Ejemplo: quitar duplicados de un arreglo ordenado.

In [ ]:
def sin_duplicados(arr):
    if not arr:
        return 0
    k = 1                              # k = cuántos únicos llevamos
    for i in range(1, len(arr)):
        if arr[i] != arr[k - 1]:
            arr[k] = arr[i]
            k += 1
    return k


a = [1, 1, 2, 2, 2, 3]
k = sin_duplicados(a)
assert k == 3 and a[:k] == [1, 2, 3]

### Variante C: mezclar dos arreglos ordenados

Un puntero en cada arreglo; avanza el del menor. Es la base de *merge sort*.

In [ ]:
def mezclar(a, b):
    i = j = 0
    res = []
    while i < len(a) and j < len(b):
        if a[i] <= b[j]:
            res.append(a[i])
            i += 1
        else:
            res.append(b[j])
            j += 1
    res.extend(a[i:])
    res.extend(b[j:])
    return res


assert mezclar([1, 4, 7], [2, 3, 9, 10]) == [1, 2, 3, 4, 7, 9, 10]

### Variante D: contar pares con suma menor o igual a `k`

In [ ]:
def pares_suma_menor_igual(arr, k):
    arr = sorted(arr)
    i, j = 0, len(arr) - 1
    total = 0
    while i < j:
        if arr[i] + arr[j] <= k:
            total += j - i          # arr[i] forma par con todos entre i+1 y j
            i += 1
        else:
            j -= 1
    return total


assert pares_suma_menor_igual([1, 2, 3, 4], 5) == 4      # (1,2) (1,3) (1,4) (2,3)

## 3. Ventana deslizante

**Idea:** mantener una ventana `[izq, der]` sobre un arreglo (o string). Al avanzar, **agregas** el nuevo elemento y **quitas** el que sale, sin recalcular todo.

### Ventana de tamaño fijo

Mayor suma de `k` elementos consecutivos.

In [ ]:
def mayor_suma_ventana(arr, k):
    ventana = sum(arr[:k])
    mejor = ventana
    for i in range(k, len(arr)):
        ventana += arr[i] - arr[i - k]        # entra arr[i], sale arr[i-k]
        mejor = max(mejor, ventana)
    return mejor


assert mayor_suma_ventana([2, 1, 5, 1, 3, 2], 3) == 9        # 5+1+3
assert mayor_suma_ventana([4], 1) == 4

### Ventana de tamaño variable

Se **expande** la derecha siempre y se **contrae** la izquierda cuando la ventana deja de cumplir la condición.

Problema: longitud del subarreglo más largo (de números **positivos**) con suma `≤ k`.

In [ ]:
def subarreglo_largo_suma_max(arr, k):
    izq = 0
    suma = 0
    mejor = 0
    for der, x in enumerate(arr):
        suma += x
        while suma > k:                 # contraer
            suma -= arr[izq]
            izq += 1
        mejor = max(mejor, der - izq + 1)
    return mejor


assert subarreglo_largo_suma_max([2, 1, 3, 1, 1], 5) == 3       # 1+3+1 o 3+1+1
assert subarreglo_largo_suma_max([5, 6], 4) == 0

Problema: la subcadena **más larga sin letras repetidas**.

In [ ]:
def subcadena_sin_repetir(s):
    ultima = {}                         # última posición de cada letra
    izq = 0
    mejor = 0
    for der, c in enumerate(s):
        if c in ultima and ultima[c] >= izq:
            izq = ultima[c] + 1         # salta justo después de la repetida
        ultima[c] = der
        mejor = max(mejor, der - izq + 1)
    return mejor


assert subcadena_sin_repetir("abcabcbb") == 3
assert subcadena_sin_repetir("bbbbb") == 1
assert subcadena_sin_repetir("") == 0

Problema: el menor subarreglo cuya suma es **al menos** `k` (positivos).

In [ ]:
def menor_subarreglo_suma_min(arr, k):
    izq = 0
    suma = 0
    mejor = float("inf")
    for der, x in enumerate(arr):
        suma += x
        while suma >= k:
            mejor = min(mejor, der - izq + 1)
            suma -= arr[izq]
            izq += 1
    return 0 if mejor == float("inf") else mejor


assert menor_subarreglo_suma_min([2, 3, 1, 2, 4, 3], 7) == 2        # 4+3
assert menor_subarreglo_suma_min([1, 1], 5) == 0

## ¿Qué técnica elijo?

| Si el problema dice... | Prueba con |
|---|---|
| muchas consultas de suma en rangos | **sumas de prefijos** |
| sumar un valor a un rango muchas veces | **diferencias** |
| pares que cumplen algo, arreglo ordenado | **dos punteros** |
| mezclar o comparar dos secuencias ordenadas | **dos punteros** |
| subarreglo/subcadena consecutiva con una condición | **ventana deslizante** |
| el mejor bloque de tamaño `k` | **ventana fija** |

⚠️ La ventana variable con "contraer cuando se pasa" solo funciona si **agregar elementos solo empeora** la condición (por ejemplo, números positivos y suma máxima).

## Problemas tipo (estilo grader)

**P1.** `equilibrio(arr)`: índice donde la suma de la izquierda es igual a la de la derecha (o `-1`).
**P2.** `triples_suma_cero(arr)`: cuántas ternas de posiciones distintas `(i, j, k)` suman 0 (dos punteros dentro de un `for`).

In [ ]:
def equilibrio(arr):
    total = sum(arr)
    izq = 0
    for i, x in enumerate(arr):
        if izq == total - izq - x:
            return i
        izq += x
    return -1


def triples_suma_cero(arr):
    arr = sorted(arr)
    total = 0
    n = len(arr)
    for i in range(n - 2):
        j, k = i + 1, n - 1
        while j < k:
            s = arr[i] + arr[j] + arr[k]
            if s == 0:
                total += 1
                j += 1
                k -= 1
            elif s < 0:
                j += 1
            else:
                k -= 1
    return total


assert equilibrio([1, 7, 3, 6, 5, 6]) == 3
assert equilibrio([1, 2, 3]) == -1
assert triples_suma_cero([-1, 0, 1, 2, -1, -4]) == 3       # cuenta con repetidos de posición
print("ok")

## Para practicar

1. `maximo_en_cada_ventana(arr, k)`: el máximo de cada ventana de tamaño `k` (con `deque`, ver cuaderno 06).
2. `contar_subarreglos_suma_menor(arr, k)` (positivos, ventana variable).
3. `pares_con_diferencia(arr, d)`: cuántos pares tienen diferencia `d` (dos punteros sobre arreglo ordenado).
4. `subcadena_con_k_distintos(s, k)`: la más larga con a lo sumo `k` letras distintas.
5. `cuadrado_mayor_suma(m, k)`: el bloque `k x k` de mayor suma (prefijos 2D).

## Errores típicos

- Desfase de uno con `pref[r + 1] - pref[l]`.
- Olvidar que **dos punteros** sobre pares requiere el arreglo **ordenado**.
- No actualizar la ventana al mover `izq`.
- Usar ventana variable con números negativos (la condición ya no es monótona: usa prefijos con diccionario).